In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
base_dir = Path.cwd()

upload_dir = base_dir / "Uploads"
submissions_dir = base_dir / "Submissions"
experiments_dir = base_dir / "leaderboard_experiments"

experiments_dir.mkdir(parents=True, exist_ok=True)

print("Upload dir:", upload_dir)
print("Submissions dir:", submissions_dir)
print("Experiments dir:", experiments_dir)

Upload dir: /home/sagemaker-user/Uploads
Submissions dir: /home/sagemaker-user/Submissions
Experiments dir: /home/sagemaker-user/leaderboard_experiments


In [12]:
v61_file = upload_dir / "forecast_v61_VIRAL.csv"
v19_file = submissions_dir / "forecast_v19.csv"

v61 = pd.read_csv(v61_file)
v19 = pd.read_csv(v19_file)

print("v61 shape:", v61.shape)
print("v19 shape:", v19.shape)

display(v61.head(3))
display(v19.head(3))

v61 shape: (1488, 19)
v19 shape: (1488, 19)


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,0:00,7,0,0.0,393.03,41,0,0.0,331.83,88,0,0.000000,356.48,47,0,0.000000,326.92
1,August,1,0:30,5,0,0.0,346.74,30,0,0.0,308.78,61,1,0.016393,316.86,33,0,0.000000,299.26
2,August,1,1:00,4,0,0.0,323.62,13,0,0.0,433.65,43,3,0.069767,327.43,24,1,0.041667,363.78


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,0:00,2.22,0.0,0.0005,311.15,6.72,0.0,0.0007,331.82,23.44,0.01,0.0005,327.78,10.47,0.01,0.0006,312.14
1,August,1,0:30,2.22,0.0,0.0005,311.15,6.72,0.0,0.0007,331.82,23.44,0.01,0.0005,327.78,10.47,0.01,0.0006,312.14
2,August,1,1:00,2.22,0.0,0.0005,311.15,6.72,0.0,0.0007,331.82,23.44,0.01,0.0005,327.78,10.47,0.01,0.0006,312.14


In [13]:
print("Same columns:", list(v61.columns) == list(v19.columns))
print("v61 columns:", v61.columns.tolist())

Same columns: True
v61 columns: ['Month', 'Day', 'Interval', 'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A', 'Calls_Offered_B', 'Abandoned_Calls_B', 'Abandoned_Rate_B', 'CCT_B', 'Calls_Offered_C', 'Abandoned_Calls_C', 'Abandoned_Rate_C', 'CCT_C', 'Calls_Offered_D', 'Abandoned_Calls_D', 'Abandoned_Rate_D', 'CCT_D']


In [14]:
id_cols = ["Month", "Day", "Interval"]

metric_cols = [c for c in v61.columns if c not in id_cols]
portfolio_suffixes = ["A", "B", "C", "D"]

calls_cols = [f"Calls_Offered_{p}" for p in portfolio_suffixes]
abd_calls_cols = [f"Abandoned_Calls_{p}" for p in portfolio_suffixes]
abd_rate_cols = [f"Abandoned_Rate_{p}" for p in portfolio_suffixes]
cct_cols = [f"CCT_{p}" for p in portfolio_suffixes]

print("Calls cols:", calls_cols)
print("Abandon rate cols:", abd_rate_cols)
print("Abandon calls cols:", abd_calls_cols)
print("CCT cols:", cct_cols)

Calls cols: ['Calls_Offered_A', 'Calls_Offered_B', 'Calls_Offered_C', 'Calls_Offered_D']
Abandon rate cols: ['Abandoned_Rate_A', 'Abandoned_Rate_B', 'Abandoned_Rate_C', 'Abandoned_Rate_D']
Abandon calls cols: ['Abandoned_Calls_A', 'Abandoned_Calls_B', 'Abandoned_Calls_C', 'Abandoned_Calls_D']
CCT cols: ['CCT_A', 'CCT_B', 'CCT_C', 'CCT_D']


In [15]:
def enforce_non_negative(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)
    return df

In [16]:
def recompute_abandoned_calls(df):
    df = df.copy()
    for p in portfolio_suffixes:
        cv_col = f"Calls_Offered_{p}"
        ar_col = f"Abandoned_Rate_{p}"
        ac_col = f"Abandoned_Calls_{p}"

        df[ac_col] = df[cv_col] * df[ar_col]
    return df

In [17]:
def round_submission(df):
    df = df.copy()
    
    # keep IDs as-is
    for col in calls_cols + abd_calls_cols + abd_rate_cols + cct_cols:
        if col in df.columns:
            df[col] = df[col].astype(float)
    
    return df

In [21]:
def validate_submission_basic(df, name="file"):
    print(f"\nValidating {name}")
    print("Shape:", df.shape)
    print("Negative values present:", (df[metric_cols] < 0).any().any())
    
    for p in portfolio_suffixes:
        cv_col = f"Calls_Offered_{p}"
        ar_col = f"Abandoned_Rate_{p}"
        ac_col = f"Abandoned_Calls_{p}"
        
        gap = (df[ac_col] - (df[cv_col] * df[ar_col])).abs()
        print(f"{p} mean abandon gap:", gap.mean(), "| max gap:", gap.max())

In [22]:
def save_submission(df, filename):
    path = experiments_dir / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

In [23]:
print("Helper function cells ran successfully.")
print(validate_submission_basic)
print(save_submission)

Helper function cells ran successfully.
<function validate_submission_basic at 0x7f78478834c0>
<function save_submission at 0x7f78478825c0>


In [24]:
def blend_files(df1, df2, w1=0.8, w2=0.2):
    df = df1[id_cols].copy()

    for col in calls_cols:
        df[col] = w1 * df1[col] + w2 * df2[col]

    for col in cct_cols:
        df[col] = w1 * df1[col] + w2 * df2[col]

    for col in abd_rate_cols:
        df[col] = w1 * df1[col] + w2 * df2[col]

    df = recompute_abandoned_calls(df)
    df = enforce_non_negative(df, calls_cols + abd_calls_cols + abd_rate_cols + cct_cols)
    df = round_submission(df)

    return df

In [25]:
ensemble_90_10 = blend_files(v61, v19, 0.90, 0.10)
ensemble_80_20 = blend_files(v61, v19, 0.80, 0.20)
ensemble_70_30 = blend_files(v61, v19, 0.70, 0.30)

validate_submission_basic(ensemble_90_10, "ensemble_90_10")
validate_submission_basic(ensemble_80_20, "ensemble_80_20")
validate_submission_basic(ensemble_70_30, "ensemble_70_30")


Validating ensemble_90_10
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0

Validating ensemble_80_20
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0

Validating ensemble_70_30
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0


In [26]:
def smart_hybrid(df_vol, df_cct):
    df = df_vol[id_cols].copy()

    # TAKE volume from best file (likely v61)
    for col in calls_cols:
        df[col] = df_vol[col]

    # TAKE CCT from stable file (maybe v19)
    for col in cct_cols:
        df[col] = df_cct[col]

    # TAKE abandon rate (try both later)
    for col in abd_rate_cols:
        df[col] = df_vol[col]

    df = recompute_abandoned_calls(df)
    df = enforce_non_negative(df, calls_cols + abd_calls_cols + abd_rate_cols + cct_cols)
    df = round_submission(df)

    return df

In [27]:
hybrid_1 = smart_hybrid(v61, v19)
hybrid_2 = smart_hybrid(v19, v61)

In [28]:
validate_submission_basic(hybrid_1, "hybrid_1")
validate_submission_basic(hybrid_2, "hybrid_2")


Validating hybrid_1
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0

Validating hybrid_2
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0


In [29]:
def row_wise_best_volume(df1, df2):
    df = df1[id_cols].copy()

    for col in calls_cols:
        df[col] = np.maximum(df1[col], df2[col])

    for col in cct_cols:
        df[col] = (df1[col] + df2[col]) / 2

    for col in abd_rate_cols:
        df[col] = (df1[col] + df2[col]) / 2

    df = recompute_abandoned_calls(df)
    df = enforce_non_negative(df, calls_cols + abd_calls_cols + abd_rate_cols + cct_cols)
    df = round_submission(df)

    return df

In [30]:
def peak_boost_strategy(df1, df2, boost_factor=1.05):
    df = df1[id_cols].copy()

    for col in calls_cols:
        base = np.maximum(df1[col], df2[col])

        # detect peaks (top 30% values)
        threshold = np.percentile(base, 70)

        boosted = base.copy()
        boosted[base >= threshold] = base[base >= threshold] * boost_factor

        df[col] = boosted

    # CCT → take safer one (lower usually better)
    for col in cct_cols:
        df[col] = np.minimum(df1[col], df2[col])

    # Abandon rate → keep stable
    for col in abd_rate_cols:
        df[col] = (df1[col] + df2[col]) / 2

    df = recompute_abandoned_calls(df)
    df = enforce_non_negative(df, calls_cols + abd_calls_cols + abd_rate_cols + cct_cols)
    df = round_submission(df)

    return df

In [31]:
peak_model = peak_boost_strategy(v61, v19, boost_factor=1.05)

validate_submission_basic(peak_model, "peak_model")


Validating peak_model
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0


In [32]:
peak_103 = peak_boost_strategy(v61, v19, 1.03)
peak_105 = peak_boost_strategy(v61, v19, 1.05)
peak_108 = peak_boost_strategy(v61, v19, 1.08)

In [33]:
validate_submission_basic(peak_103, "peak_103")
validate_submission_basic(peak_105, "peak_105")
validate_submission_basic(peak_108, "peak_108")


Validating peak_103
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0

Validating peak_105
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0

Validating peak_108
Shape: (1488, 19)
Negative values present: False
A mean abandon gap: 0.0 | max gap: 0.0
B mean abandon gap: 0.0 | max gap: 0.0
C mean abandon gap: 0.0 | max gap: 0.0
D mean abandon gap: 0.0 | max gap: 0.0


In [34]:
def compare_to_base(base_df, new_df, name):
    rows = []

    for p in portfolio_suffixes:
        cv_col = f"Calls_Offered_{p}"
        ar_col = f"Abandoned_Rate_{p}"
        ac_col = f"Abandoned_Calls_{p}"
        cct_col = f"CCT_{p}"

        rows.append({
            "candidate": name,
            "portfolio": p,
            "mean_calls_diff": (new_df[cv_col] - base_df[cv_col]).mean(),
            "pct_calls_changed": ((new_df[cv_col] != base_df[cv_col]).mean()) * 100,
            "mean_abd_rate_diff": (new_df[ar_col] - base_df[ar_col]).mean(),
            "mean_abd_calls_diff": (new_df[ac_col] - base_df[ac_col]).mean(),
            "mean_cct_diff": (new_df[cct_col] - base_df[cct_col]).mean()
        })

    return pd.DataFrame(rows)

comp_103 = compare_to_base(v61, peak_103, "peak_103")
comp_105 = compare_to_base(v61, peak_105, "peak_105")
comp_108 = compare_to_base(v61, peak_108, "peak_108")

display(comp_103)
display(comp_105)
display(comp_108)

,candidate,portfolio,mean_calls_diff,pct_calls_changed,mean_abd_rate_diff,mean_abd_calls_diff,mean_cct_diff
0,peak_103,A,9.045415,65.524194,-0.001041,0.114900,-14.051351
1,peak_103,B,17.959274,62.970430,-0.003765,0.387604,-12.832392
2,peak_103,C,35.348261,64.314516,-0.004427,-0.585726,-8.518871
3,peak_103,D,20.004113,59.677419,-0.002824,-0.195084,-13.479993


,candidate,portfolio,mean_calls_diff,pct_calls_changed,mean_abd_rate_diff,mean_abd_calls_diff,mean_cct_diff
0,peak_105,A,10.284642,65.524194,-0.001041,0.120143,-14.051351
1,peak_105,B,20.472187,62.970430,-0.003765,0.427916,-12.832392
2,peak_105,C,40.743588,64.314516,-0.004427,-0.498020,-8.518871
3,peak_105,D,22.767370,59.677419,-0.002824,-0.151922,-13.479993


,candidate,portfolio,mean_calls_diff,pct_calls_changed,mean_abd_rate_diff,mean_abd_calls_diff,mean_cct_diff
0,peak_108,A,12.143484,65.524194,-0.001041,0.128007,-14.051351
1,peak_108,B,24.241555,62.970430,-0.003765,0.488386,-12.832392
2,peak_108,C,48.836580,64.314516,-0.004427,-0.366461,-8.518871
3,peak_108,D,26.912256,59.677419,-0.002824,-0.087179,-13.479993


In [35]:
save_submission(peak_103, "forecast_peak_103.csv")
save_submission(peak_105, "forecast_peak_105.csv")
save_submission(peak_108, "forecast_peak_108.csv")

Saved: /home/sagemaker-user/leaderboard_experiments/forecast_peak_103.csv
Saved: /home/sagemaker-user/leaderboard_experiments/forecast_peak_105.csv
Saved: /home/sagemaker-user/leaderboard_experiments/forecast_peak_108.csv


PosixPath('/home/sagemaker-user/leaderboard_experiments/forecast_peak_108.csv')